# ML Factory - Trading Ensemble Builder

Run the complete ML Factory pipeline from data to deployment.

**How to use:**
1. Run Cell 1 (Setup) once
2. Edit Cell 2 (Configuration) with your choices
3. Run all remaining cells

**Requirements:** Google Colab with GPU runtime recommended for neural/transformer models.

In [ ]:
# =============================================================
# CELL 1: SETUP - Run this once
# =============================================================
# Mount Google Drive for data and saving results
from google.colab import drive
drive.mount('/content/drive')

# Clone ML Factory repository
import os
import sys

REPO_DIR = "/content/ml_factory"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Snehpatel101/Research.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet

# Install dependencies
!cd {REPO_DIR} && pip install -q -r requirements-colab.txt 2>&1 | tail -1

# Add to Python path
sys.path.insert(0, REPO_DIR)

# Verify installation
try:
    from src.factory import MLFactory
    from src.config.experiment import ExperimentConfig
    print("ML Factory loaded successfully!")
except ImportError as e:
    print(f"ERROR: {e}")
    print("Check that the repository cloned correctly.")

# GPU check
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Boosting models work fine on CPU.")
    print("For neural/transformer models, enable GPU: Runtime > Change runtime type > GPU")

In [ ]:
# =============================================================
# CELL 2: CONFIGURATION - Edit these settings
# =============================================================
# Change any values below. Defaults are sensible starting points.

# ----- SECTION 1: YOUR DATA -----
SYMBOL = "MES"                          # Trading symbol (e.g., "MES", "ES", "NQ", "SPY")
DATA_PATH = "/content/drive/MyDrive/data/MES_5min.parquet"  # Path to your OHLCV data file

# ----- SECTION 2: PICK YOUR MODELS -----
# Speed tiers (approximate training time per horizon):
#   FAST   (~1-3 min):  "xgboost", "lightgbm", "catboost"
#   MEDIUM (~5-15 min): "lstm", "gru", "nbeats"
#   SLOW   (~15-45 min): "tcn", "inception_time", "resnet_1d"
#   GPU-HEAVY (~30-90 min): "patchtst", "itransformer", "tft"
MODELS = [
    "xgboost",
    "lightgbm",
    # "catboost",
    # "lstm",
    # "gru",
    # "tcn",
    # "inception_time",
    # "resnet_1d",
    # "nbeats",
    # "patchtst",
    # "itransformer",
    # "tft",
]

# ----- SECTION 3: WHAT TO OPTIMIZE FOR -----
# Options: "sharpe_ratio", "f1_weighted", "accuracy", "sortino_ratio", "profit_factor"
# This metric is used by the optimizer during hyperparameter tuning and feature selection.
# Trading metrics (sharpe, sortino, profit_factor) use a classification proxy.
OPTIMIZE_FOR = "sharpe_ratio"

# ----- SECTION 4: PREDICTION HORIZONS -----
# Bars ahead to predict. More horizons = longer training.
HORIZONS = [5, 10, 15, 20]

# ----- SECTION 5: ENSEMBLE SETTINGS -----
BUILD_ENSEMBLE = True                   # Combine models into an ensemble
META_LEARNER = "ridge_meta"             # Meta-learner: "ridge_meta"

# ----- SECTION 6: LABELING METHOD -----
# Options: "triple_barrier", "directional", "threshold"
LABELING_METHOD = "triple_barrier"

# ----- SECTION 7: FEATURE FAMILIES -----
# Core families (recommended):
FEATURE_FAMILIES = [
    "price",
    "momentum",
    "volatility",
    "volume",
    "trend",
    # --- Advanced (uncomment to enable) ---
    # "microstructure",
    # "regime",
    # "wavelet",
]

# ----- SECTION 8: BACKTEST & EVALUATION -----
RUN_BACKTEST = True                     # Run walk-forward backtest after training
# Position sizing: "fixed", "kelly", "volatility", "confidence"
POSITION_SIZING = "fixed"

# ----- SECTION 9: TUNING INTENSITY -----
# More trials = better hyperparameters but longer runtime
# ~25 trials: 5-10 min per model   (quick test)
# ~50 trials: 10-20 min per model  (reasonable)
# ~100 trials: 20-45 min per model (production)
OPTUNA_TRIALS = 50

# ----- SECTION 10: GENERAL -----
EXPERIMENT_NAME = "my_experiment"       # Name for this experiment run
RANDOM_SEED = 42                        # For reproducibility
SAVE_TO_DRIVE = True                    # Save results to Google Drive

# =============================================================
# SUMMARY
# =============================================================
print("=" * 50)
print("ML Factory Configuration Summary")
print("=" * 50)
print(f"  Models:        {len(MODELS)} selected - {', '.join(MODELS)}")
print(f"  Ensemble:      {'Yes' if BUILD_ENSEMBLE else 'No'}")
print(f"  Optimize for:  {OPTIMIZE_FOR}")
print(f"  Horizons:      {HORIZONS}")
print(f"  Optuna trials: {OPTUNA_TRIALS}")
print(f"  Labeling:      {LABELING_METHOD}")
print(f"  Backtest:      {'Yes' if RUN_BACKTEST else 'No'}")
print("=" * 50)

In [ ]:
# =============================================================
# CELL 3: VALIDATION - Run to check your configuration
# =============================================================
import os

VALID_MODELS = {
    "xgboost", "lightgbm", "catboost",
    "lstm", "gru",
    "tcn", "inception_time", "resnet_1d",
    "patchtst", "itransformer", "tft",
    "nbeats",
}

NEURAL_MODELS = {
    "lstm", "gru", "tcn", "inception_time", "resnet_1d",
    "patchtst", "itransformer", "tft", "nbeats",
}

errors = []
warnings = []

# Validate model names
for m in MODELS:
    if m not in VALID_MODELS:
        errors.append(f"Unknown model: '{m}'. Valid: {sorted(VALID_MODELS)}")

if not MODELS:
    errors.append("MODELS list is empty. Select at least one model.")

# Check data path
if not os.path.exists(DATA_PATH):
    warnings.append(f"DATA_PATH not found: {DATA_PATH}  (will fail at pipeline step)")

# GPU check for neural models
selected_neural = [m for m in MODELS if m in NEURAL_MODELS]
if selected_neural:
    import torch
    if not torch.cuda.is_available():
        warnings.append(
            f"GPU not available but neural models selected: {selected_neural}. "
            "Training will be very slow. Enable GPU: Runtime > Change runtime type > GPU"
        )

# Print results
if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise ValueError("Configuration has errors. Fix them above and re-run.")

if warnings:
    for w in warnings:
        print(f"WARNING: {w}")

print("Configuration validated.")

In [ ]:
# =============================================================
# CELL 4: LOAD & PREVIEW DATA
# =============================================================
import pandas as pd

# Load data based on file extension
if DATA_PATH.endswith(".parquet"):
    raw_data = pd.read_parquet(DATA_PATH)
elif DATA_PATH.endswith(".csv"):
    raw_data = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH}. Use .parquet or .csv")

# Normalize column names to lowercase
raw_data.columns = [c.lower().strip() for c in raw_data.columns]

# Validate required OHLCV columns
REQUIRED_COLUMNS = ["open", "high", "low", "close", "volume"]
missing = [c for c in REQUIRED_COLUMNS if c not in raw_data.columns]
if missing:
    raise ValueError(
        f"Missing required OHLCV columns: {missing}\n"
        f"Found columns: {list(raw_data.columns)}\n"
        f"The pipeline expects: {REQUIRED_COLUMNS}"
    )

# Ensure datetime index
if "datetime" in raw_data.columns:
    raw_data["datetime"] = pd.to_datetime(raw_data["datetime"])
    raw_data = raw_data.set_index("datetime").sort_index()
elif "date" in raw_data.columns:
    raw_data["date"] = pd.to_datetime(raw_data["date"])
    raw_data = raw_data.set_index("date").sort_index()
    raw_data.index.name = "datetime"
elif not isinstance(raw_data.index, pd.DatetimeIndex):
    # Try parsing the existing index as datetime
    try:
        raw_data.index = pd.to_datetime(raw_data.index)
        raw_data.index.name = "datetime"
        raw_data = raw_data.sort_index()
    except Exception:
        raise ValueError(
            "Could not find or parse a datetime column. "
            "Data must have a 'datetime' or 'date' column, or a datetime-parseable index."
        )

# --- Summary ---
print("=" * 50)
print("Data Loaded Successfully")
print("=" * 50)
print(f"  Symbol:      {SYMBOL}")
print(f"  Rows:        {len(raw_data):,}")
print(f"  Shape:       {raw_data.shape}")
print(f"  Columns:     {list(raw_data.columns)}")
print(f"  Date range:  {raw_data.index.min()} -> {raw_data.index.max()}")
print(f"  Index name:  {raw_data.index.name}")
print()

# Missing values
missing_counts = raw_data[REQUIRED_COLUMNS].isnull().sum()
total_missing = missing_counts.sum()
if total_missing > 0:
    print("WARNING: Missing values in OHLCV columns:")
    for col, count in missing_counts.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(raw_data)*100:.2f}%)")
else:
    print("No missing values in OHLCV columns.")
print()

# Preview
print("First 5 rows:")
display(raw_data.head())

print(f"\nData ready: 'raw_data' DataFrame with {len(raw_data):,} rows.")

In [ ]:
# =============================================================
# CELL 5: ASSEMBLE CONFIG & RUN FACTORY
# =============================================================
from src.config.experiment import (
    ExperimentConfig,
    DataSection,
    TrainingSection,
    EvaluationSection,
)
from src.config.training import OptunaConfig
from src.config.data import FeatureConfig, LabelingConfig
from src.factory import MLFactory

# --- Assemble configuration from Cell 2 variables ---
config = ExperimentConfig(
    name=EXPERIMENT_NAME,
    random_seed=RANDOM_SEED,
    data=DataSection(
        symbol=SYMBOL,
        data_path=DATA_PATH,
        features=FeatureConfig(families=FEATURE_FAMILIES),
        labeling=LabelingConfig(method=LABELING_METHOD),
    ),
    training=TrainingSection(
        models=MODELS,
        horizons=HORIZONS,
        build_ensemble=BUILD_ENSEMBLE,
        meta_learner=META_LEARNER,
        optuna=OptunaConfig(
            n_trials=OPTUNA_TRIALS,
            metric=OPTIMIZE_FOR,
        ),
    ),
    evaluation=EvaluationSection(
        run_backtest=RUN_BACKTEST,
        position_sizing=POSITION_SIZING,
    ),
)

print("ExperimentConfig assembled:")
print(f"  Name:           {config.name}")
print(f"  Symbol:         {config.data.symbol}")
print(f"  Data path:      {config.data.data_path}")
print(f"  Models:         {config.training.models}")
print(f"  Horizons:       {config.training.horizons}")
print(f"  Ensemble:       {config.training.build_ensemble}")
print(f"  Meta-learner:   {config.training.meta_learner}")
print(f"  Optimize for:   {config.training.optuna.metric}")
print(f"  Optuna trials:  {config.training.optuna.n_trials}")
print(f"  Labeling:       {config.data.labeling.method}")
print(f"  Features:       {config.data.features.families}")
print(f"  Backtest:       {config.evaluation.run_backtest}")
print(f"  Pos. sizing:    {config.evaluation.position_sizing}")
print(f"  Random seed:    {config.random_seed}")
print(f"  Output dir:     {config.output_dir}")
print()

# --- Run the factory ---
factory = MLFactory(config, enable_checkpoints=True)

try:
    print("Starting ML Factory pipeline...")
    print("This may take a while depending on model selection and Optuna trials.")
    print()
    result = factory.run()
    print()
    print(result.summary())
except Exception as e:
    print()
    print(f"ERROR: Factory run failed: {e}")
    print()
    print("To resume from the last checkpoint, run:")
    print("  result = factory.resume_from_checkpoint()")
    print()
    print("Or fix the issue above and re-run this cell.")
    result = None

In [ ]:
# =============================================================
# CELL 6: RESULTS & VISUALIZATION
# =============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob

if result is None or not result.success:
    msg = "No successful result to display."
    if result and result.error_message:
        msg += f"\nError: {result.error_message}"
    print(msg)
else:
    print("=" * 60)
    print("EXPERIMENT RESULTS")
    print("=" * 60)
    print(f"  Run ID:          {result.run_id}")
    print(f"  Models trained:  {result.n_models}")
    print(f"  Best model:      {result.best_model}")
    print(f"  Duration:        {result.duration_seconds:.1f}s ({result.duration_seconds/60:.1f} min)")
    print()

    # --- Model Metrics Table ---
    if result.metrics:
        print("-" * 40)
        print("Model Performance")
        print("-" * 40)
        metrics_df = pd.DataFrame(result.metrics).T
        metrics_df.index.name = "model"
        display(metrics_df.round(4))
        print()

    # --- Ensemble Metrics ---
    if result.ensemble_metrics:
        print("-" * 40)
        print("Ensemble Metrics")
        print("-" * 40)
        for k, v in result.ensemble_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        print()

    # --- Backtest Metrics ---
    if result.backtest_metrics:
        print("-" * 40)
        print("Backtest Results")
        print("-" * 40)
        highlight_keys = ["sharpe_ratio", "max_drawdown", "profit_factor", "win_rate_pct"]
        for k in highlight_keys:
            if k in result.backtest_metrics:
                print(f"  {k}: {result.backtest_metrics[k]}")
        for k, v in result.backtest_metrics.items():
            if k not in highlight_keys:
                if isinstance(v, (int, float)):
                    print(f"  {k}: {v}")
        print()

    # --- Display Plot Images ---
    if result.output_dir and Path(result.output_dir).exists():
        plot_files = sorted(glob.glob(str(Path(result.output_dir) / "**" / "*.png"), recursive=True))[:6]
        if plot_files:
            print("-" * 40)
            print(f"Plots ({len(plot_files)} found)")
            print("-" * 40)
            n_plots = len(plot_files)
            cols = min(n_plots, 2)
            rows = (n_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
            if n_plots == 1:
                axes = [axes]
            else:
                axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, pf in enumerate(plot_files):
                img = mpimg.imread(pf)
                axes[i].imshow(img)
                axes[i].set_title(Path(pf).stem, fontsize=10)
                axes[i].axis("off")
            # Hide unused subplots
            for j in range(n_plots, len(axes)):
                axes[j].axis("off")
            plt.tight_layout()
            plt.show()

    if result.bundle_path:
        print(f"Bundle path: {result.bundle_path}")
    if result.output_dir:
        print(f"Output dir:  {result.output_dir}")

In [ ]:
# =============================================================
# CELL 7: SAVE RESULTS TO GOOGLE DRIVE
# =============================================================
import shutil
from pathlib import Path

if not SAVE_TO_DRIVE:
    print("SAVE_TO_DRIVE is False. Skipping save.")
elif result is None or not result.success:
    print("No successful result to save.")
elif result.output_dir and Path(result.output_dir).exists():
    drive_dest = Path(f"/content/drive/MyDrive/ml_factory_results/{EXPERIMENT_NAME}")
    drive_dest.mkdir(parents=True, exist_ok=True)

    src_dir = Path(result.output_dir)
    print(f"Copying results to Google Drive...")
    print(f"  Source:      {src_dir}")
    print(f"  Destination: {drive_dest}")

    # Copy entire output directory
    if drive_dest.exists():
        shutil.rmtree(drive_dest)
    shutil.copytree(src_dir, drive_dest)

    # Summary
    copied_files = list(drive_dest.rglob("*"))
    n_files = sum(1 for f in copied_files if f.is_file())
    print()
    print("=" * 50)
    print("Save Complete")
    print("=" * 50)
    print(f"  Files copied:    {n_files}")
    print(f"  Drive location:  {drive_dest}")
    if result.bundle_path:
        bundle_in_drive = drive_dest / Path(result.bundle_path).relative_to(src_dir) if src_dir in Path(result.bundle_path).parents else result.bundle_path
        print(f"  Bundle:          {bundle_in_drive}")
    print(f"  Experiment:      {EXPERIMENT_NAME}")
    print()
    print("Results saved. You can access them at:")
    print(f"  {drive_dest}")
else:
    print("No output directory found in result. Nothing to save.")